# Next Optimization Prompt (V177)

You are Kaggle GrandMaster working on NeuroGolf 2026. Continue from **V176**, whose mother baseline is the user-provided `submission (17).zip` plus one verified local graph surgery on task233. Do **not** continue from V171/V173/V174 if a newer user-provided submission zip exists. If the user uploads another `submission*.zip`, treat it as the new highest baseline, compare per-task cost/correctness, and only graft task-level ONNX files or graph-surgery mutations that pass full public ARC + ARC-GEN validation and reduce cost.

Next round priorities:
1. Re-score V176 against the latest uploaded baseline with the official or equivalent scorer under ORT optimizations disabled.
2. Target high-cost tasks first: task233, task366, task158, task286, task054, task018, task133, task285, task367, task187, task349, task364, task002, task101, task243.
3. Search only low-risk local rewrites first: redundant `Where` bypass, Cast bypass with exact output equivalence, dead initializer removal, duplicate initializer merge, safe scalar/shape constant reduction, and node-output memory elimination only when ONNX checker + ORT 1.24.4 both accept it.
4. For every candidate, record task id, mutation, old cost, new cost, public sample count, exact output equality against the parent model, target correctness, and rollback path.
5. Keep the notebook source below 1MB, keep markdown in English, and always emit `/kaggle/working/submission.zip`.


# V176 Optimization Notes

Mother baseline: user-provided `submission (17).zip`, which already improved slightly over `submission (16).zip` on task005, task014, and task131.

This version adds one verified improvement over `submission (17).zip`:

| Task | Mutation | Validation | Cost change |
|---:|---|---:|---:|
| 233 | Safe `Where` bypass at node 203 input 1 | 266 / 266 public ARC + ARC-GEN examples passed; raw output exactly matched the parent model on all public examples | 32011 → 31975 |

Total static scorer proxy change versus `submission (17).zip`: **-36 cost**. This is a deliberately small but real improvement, not a score-retaining rebuild. The modified graph was checked with ONNX checker, strict shape inference, and ONNX Runtime with graph optimizations disabled. All 400 ONNX files are written at zip root as `task001.onnx` ... `task400.onnx`.

Rejected directions during this round:
- Broad arbitrary op bypass: many candidates lower static cost but fail exact output equivalence or runtime shape constraints.
- Scalarizing repeated initializers: tempting on task096/task280 but invalidates shape inference or changes outputs.
- Unused TopK/MaxPool output blanking: ONNX checker rejects required outputs.
- Older donor zips: no task in the available older submissions had lower static cost than `submission (17).zip`.


In [1]:
# V176 NeuroGolf submission writer and compact audit helper
# Author: 暗黑AGI / Kaggle id: boristown
# This notebook embeds the verified V176 submission zip and writes it to Kaggle working paths.

import base64
import os
import re
import zipfile
import hashlib
from pathlib import Path

SUBMISSION_B64 = """..."""
EXPECTED_SHA256 = "31c36b9157c228f8cf6886d8c02cfca27a0a9f51a46884be9cb7da5d4bd0ca87"
EXPECTED_TASK_COUNT = 400

def write_submission(out_path="/kaggle/working/submission.zip"):
    data = base64.b64decode(SUBMISSION_B64.encode("ascii"))
    sha = hashlib.sha256(data).hexdigest()
    if sha != EXPECTED_SHA256:
        raise RuntimeError(f"Embedded zip sha256 mismatch: {sha} != {EXPECTED_SHA256}")
    out = Path(out_path)
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_bytes(data)
    # Also write a local copy for notebook sessions outside Kaggle.
    Path("submission.zip").write_bytes(data)
    return str(out), len(data), sha

def audit_submission(zip_path="/kaggle/working/submission.zip"):
    with zipfile.ZipFile(zip_path) as z:
        names = sorted(n for n in z.namelist() if re.fullmatch(r"task\d{3}\.onnx", n))
        missing = [f"task{i:03d}.onnx" for i in range(1, 401) if f"task{i:03d}.onnx" not in names]
        duplicate_count = len(names) - len(set(names))
        largest = sorted(((z.getinfo(n).file_size, n) for n in names), reverse=True)[:8]
    if len(names) != EXPECTED_TASK_COUNT or missing or duplicate_count:
        raise RuntimeError({"count": len(names), "missing": missing[:10], "duplicates": duplicate_count})
    return {"count": len(names), "largest_files": largest}

path, size, sha = write_submission()
print("Wrote", path)
print("Zip bytes", size)
print("sha256", sha)
print("Audit", audit_submission(path))


Wrote /kaggle/working/submission.zip
Zip bytes 453733
sha256 31c36b9157c228f8cf6886d8c02cfca27a0a9f51a46884be9cb7da5d4bd0ca87
Audit {'count': 400, 'largest_files': [(161283, 'task243.onnx'), (111759, 'task286.onnx'), (74228, 'task018.onnx'), (39134, 'task157.onnx'), (29365, 'task366.onnx'), (27161, 'task233.onnx'), (24729, 'task054.onnx'), (17654, 'task023.onnx')]}
